# PulseIQ — Sentiment Fine-Tuning (DistilBERT + LoRA)Runs on a **Colab T4 runtime** from inside VS Code.This notebook is a thin driver. All logic lives in tested modules under`src/pulseiq/training/sentiment/` — the notebook clones the repo and calls them,so nothing here is untested notebook-only code.**Order matters:** the zero-shot baseline is scored *before* fine-tuning, on thesame held-out test set, so the before/after comparison is valid.

## 1. Verify the GPUIf this shows no GPU, reconnect and pick a T4 runtime — training on CPU is ~20x slower.

In [ ]:
!nvidia-smi

## 2. Clone the repo and installThe Colab runtime is a separate machine, so the code has to be fetched.

In [ ]:
import osREPO = "https://github.com/uditnegi16/pulseiq.git"if not os.path.exists("pulseiq"):    !git clone -q $REPO%cd pulseiq!git pull -q!pip install -q transformers peft datasets accelerate evaluate scikit-learn pyarrow pydantic-settings mlflowprint("\ninstalled")

In [ ]:
import syssys.path.insert(0, "src")sys.path.insert(0, ".")import torchprint("torch:", torch.__version__)print("cuda :", torch.cuda.is_available())if torch.cuda.is_available():    print("gpu  :", torch.cuda.get_device_name(0))

## 3. Build the datasetStreams Amazon Reviews'23 (Electronics), maps stars to binary sentiment,balances the classes, and writes train/val/test to parquet.**Balancing matters:** ~80% of Amazon reviews are 4–5 stars. Without it, a modelthat always predicts "positive" would score 80% and appear to work.**Labels are a proxy.** A 5-star review can still contain complaints, so theaccuracy ceiling is set by label noise, not model capacity. See`docs/decision-log.md` D-007.

In [ ]:
from pulseiq.training.sentiment.dataset import load_reviews, stratified_split, save_splitsfrom pathlib import Pathimport logginglogging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")MAX_ROWS = 20_000     # drop to 5_000 for a fast first passframe = load_reviews(max_rows=MAX_ROWS, balance=True)print(f"\n{len(frame)} labelled reviews")print(f"positive share: {frame['label'].mean():.1%}")print(f"median length : {frame['text'].str.split().str.len().median():.0f} words")frame.head(3)

In [ ]:
train, val, test = stratified_split(frame, test_size=0.15, val_size=0.15)paths = save_splits(train, val, test, Path("data/processed"))print(f"train={len(train)} val={len(val)} test={len(test)}")

## 4. Zero-shot baseline — the "before" number`distilbert-base-uncased-finetuned-sst-2-english`, used off the shelf. This iswhat the original project did.It is a **strong** baseline — already sentiment-tuned, just on movie reviewsrather than product reviews. A weak baseline chosen to flatter the fine-tunewould make the improvement meaningless.

In [ ]:
from pulseiq.training.sentiment.baseline_zeroshot import evaluate_baselinefrom pulseiq.evaluation.classification import to_metrics, majority_baselineimport jsonbaseline = evaluate_baseline(test, device=0 if torch.cuda.is_available() else -1)print("\nzero-shot baseline")print(to_metrics(baseline))print()print(to_metrics(baseline).confusion_table())print("\nmajority-class floor:", to_metrics(majority_baseline(test['label'])))Path("reports").mkdir(exist_ok=True)Path("reports/sentiment_baseline.json").write_text(json.dumps(baseline, indent=2))

## 5. Fine-tune with LoRALoRA freezes DistilBERT and trains small low-rank matrices in the attentionlayers — under 1% of parameters, and a few MB on disk.**Checkpoint selection uses the validation set.** The test set is scored exactlyonce, at the end. Selecting on test would be the classification equivalent ofthe temporal leakage this project fixed in forecasting.

In [ ]:
from pulseiq.training.sentiment.finetune_lora import LoRAConfig, train, evaluate_on_testconfig = LoRAConfig(    r=16,    lora_alpha=32,    learning_rate=2e-4,    batch_size=32,    epochs=3,    max_length=256,)model, tokenizer, run_info = train(train, val, config=config)print("\n", {k: v for k, v in run_info.items() if k in      ("device", "train_runtime_seconds", "trainable_params", "total_params", "adapter_size_mb")})

## 6. Score on the held-out test setFirst and only time the test set is touched by the fine-tuned model.

In [ ]:
finetuned = evaluate_on_test(model, tokenizer, test, max_length=config.max_length)print("fine-tuned (LoRA)")print(to_metrics(finetuned))print()print(to_metrics(finetuned).confusion_table())Path("reports/sentiment_finetuned.json").write_text(json.dumps({**run_info, **finetuned}, indent=2))

## 7. Before vs after`error_reduction_pct` is the honest framing at high accuracy: +3 points from 91% to 94% removes a third of the remaining errors.

In [ ]:
from pulseiq.evaluation.classification import compareimport pandas as pdcomparison = pd.DataFrame({    "zero-shot": {k: baseline[k] for k in ("accuracy","precision","recall","f1","macro_f1")},    "fine-tuned": {k: finetuned[k] for k in ("accuracy","precision","recall","f1","macro_f1")},})comparison["delta"] = comparison["fine-tuned"] - comparison["zero-shot"]print(comparison.round(4))print()for key, value in compare(baseline, finetuned).items():    print(f"  {key:<22}: {value:+.4f}")

## 8. Download the adapterThe adapter is a few MB, small enough to commit. Download it, place it in`models/sentiment_lora/` locally, and inference runs on CPU with no GPU needed.

In [ ]:
import shutilshutil.make_archive("sentiment_lora", "zip", "models/sentiment_lora")size_mb = os.path.getsize("sentiment_lora.zip") / 1e6print(f"sentiment_lora.zip — {size_mb:.2f} MB")try:    from google.colab import files    files.download("sentiment_lora.zip")except ImportError:    print("Not in the Colab web UI — right-click the file in the VS Code explorer to download.")

## 9. Log to MLflow (optional)Only works if the runtime can reach your tracking store. Skip if MLflow islocal-only — the JSON reports in `reports/` carry the numbers either way.

In [ ]:
# import mlflow# mlflow.set_tracking_uri("sqlite:///mlflow.db")# mlflow.set_experiment("pulseiq")## with mlflow.start_run(run_name="sentiment_lora_finetune"):#     mlflow.log_params({**config.as_dict(), "n_train": len(train), "n_test": len(test)})#     mlflow.log_metrics({k: v for k, v in finetuned.items() if isinstance(v, (int, float))})#     mlflow.log_metrics({f"baseline_{k}": v for k, v in baseline.items() if isinstance(v, (int, float))})

---## Next steps1. Download `sentiment_lora.zip`, unzip into `models/sentiment_lora/` locally2. Verify CPU inference: `python -m pulseiq.training.sentiment.predict --text "battery died in a week"`3. Copy the before/after table into `docs/metrics.md`4. Commit the adapter (a few MB) so the result is reproducible from the repo**When writing up:** report the majority-class floor alongside the accuracy, andstate that labels are star-derived proxies. A fine-tune that reaches ~92% wherelabels are ~95% faithful has saturated the task — pushing further would befitting label noise.